In [3]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
import scanpy.external as sce
import matplotlib.pyplot as plt

In [4]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')
# maximum number of columns
pd.options.display.max_columns = None
pd.options.display.max_rows = None

scanpy==1.9.8 anndata==0.10.5.post1 umap==0.5.5 numpy==1.26.4 scipy==1.12.0 pandas==2.2.1 scikit-learn==1.4.1.post1 statsmodels==0.14.1 igraph==0.11.4 pynndescent==0.5.11


In [5]:
adata_cd45p = sc.read_h5ad('/bigdata/zlin/PanCancer_ICI/data/TNBC_Shiao/GSE246613_PembroRT_immune_R100_final.h5ad')
adata_cd45n = sc.read_h5ad('/bigdata/zlin/PanCancer_ICI/data/TNBC_Shiao/GSE246613_PembroRT_non_immune_cells.h5ad')

In [6]:
adata_cd45n.obs.columns

Index(['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts',
       'log1p_total_counts', 'pct_counts_in_top_50_genes',
       'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes',
       'pct_counts_in_top_500_genes', 'percent_mito', 'n_counts',
       'percent_ribo', 'scrublet', 'batch', 'cohort', 'leiden',
       'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score',
       'old_celltype', 'old_subtype', 'celltype_new', 'treatment',
       'response_group', 'subtype_new'],
      dtype='object')

In [7]:
adata_cd45n.obs=adata_cd45n.obs[['batch','cohort','treatment','response_group']]

In [10]:
adata_cd45n.obs.head()

,batch,cohort,treatment,response_group
h01A_P_AAACCTGCAACTTGAC,h01A_P,Patient01,Base,R2
h01A_P_AAACCTGCATTATCTC,h01A_P,Patient01,Base,R2
h01A_P_AAACCTGGTTGTCGCG,h01A_P,Patient01,Base,R2
h01A_P_AAACCTGTCTCTGCTG,h01A_P,Patient01,Base,R2
h01A_P_AAACGGGAGAGGTTAT,h01A_P,Patient01,Base,R2


In [8]:
pd.crosstab(adata_cd45n.obs['cohort'], adata_cd45n.obs['response_group']) 

response_group,NR,R1,R2
cohort,,,
Patient01,0,0,2051
Patient02,0,689,0
Patient03T1,0,0,97
Patient03T2,0,0,482
Patient04,0,544,0
Patient05,0,1304,0
Patient06,2059,0,0
Patient07,0,639,0
Patient09,0,695,0


In [9]:
adata_cd45p.obs.head()

,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,percent_mito,n_counts,percent_ribo,scrublet,batch,CD45_enrich,batch_num,cohort,pCR,RCB,cleared_nodes,treatment,patient_treatment,hormone_receptor,combined_tcr,umap1,umap2,leiden_50nbr_res1.2,celltype,global_clusters,bcell_leiden_nbr100_res0.6,tcell_leiden_nbr100_res0.6,myeloid_leiden_nbr30_res0.8,subcluster
h01A_P_AAACCTGAGACAGACC,691,6.539586,1354.0,7.211557,37.518464,49.335303,63.737075,85.893648,0.045052,1354.0,0.127770,False,h01A_P,P,0,Patient01,R,0,Y,Base,Patient01_Base,TNBC,nan,12.238769,-7.196668,22,myeloid,12,NaN,NaN,12,myeloid_10
h01A_P_AAACCTGAGATGTGTA,3328,8.110427,15093.0,9.622052,41.065395,49.479891,58.702710,71.556351,0.007354,15093.0,0.061287,False,h01A_P,P,0,Patient01,R,0,Y,Base,Patient01_Base,TNBC,nan,5.326361,-0.320588,22,myeloid,12,NaN,NaN,08,myeloid_05
h01A_P_AAACCTGAGCGCTTAT,1675,7.424165,5568.0,8.624970,37.230603,50.053879,62.679598,77.298851,0.020654,5568.0,0.242996,False,h01A_P,P,0,Patient01,R,0,Y,Base,Patient01_Base,TNBC,nan,14.234969,0.425975,23,myeloid,9,NaN,NaN,04,myeloid_09
h01A_P_AAACCTGAGGCCCGTT,1553,7.348588,10109.0,9.221280,70.907112,75.922445,81.491740,89.069146,0.016124,10109.0,0.073796,False,h01A_P,P,0,Patient01,R,0,Y,Base,Patient01_Base,TNBC,nan,1.327492,15.696615,12,Bcell,16,06,NaN,NaN,Bcell_06
h01A_P_AAACCTGAGGTTACCT,852,6.748760,1890.0,7.544861,36.296296,49.629630,62.962963,81.375661,0.033862,1890.0,0.323280,False,h01A_P,P,0,Patient01,R,0,Y,Base,Patient01_Base,TNBC,CAMRQAGTALIF_CSARPSRQSDNEQFF,-9.247342,-2.434366,27,Tcell,1,NaN,00,NaN,Tcell_00


In [10]:
pd.crosstab(adata_cd45p.obs['batch'], adata_cd45p.obs['pCR']) 

pCR,NR,R
batch,,
h01A_P,0,8270
h01C_P,0,10799
h02A_P,0,7338
h02B_P,0,7603
h02C_P,0,3422
h03A1_P,0,1260
h03A2_P,0,1873
h03B1_P,0,1569
h03B2_P,0,3358


In [11]:
pd.crosstab(adata_cd45p.obs['cohort'], adata_cd45p.obs['pCR']) 

pCR,NR,R
cohort,,
Patient01,0,19069
Patient02,0,18363
Patient03T1,0,2855
Patient03T2,0,6755
Patient04,0,5418
Patient05,0,10270
Patient06,12363,0
Patient07,0,6455
Patient09,0,21297


In [12]:
del adata_cd45p.obsm
del adata_cd45n.obsm

In [13]:
adata_cd45p.obs.to_csv('/bigdata/zlin/Melanoma_meta/data/TNBC_Shiao/obs_cd45p.csv')
adata_cd45n.obs.to_csv('/bigdata/zlin/Melanoma_meta/data/TNBC_Shiao/obs_cd45n.csv')

In [14]:
del adata_cd45p.obs
del adata_cd45n.obs

In [15]:
diopy.output.write_h5(adata_cd45n, file = '/bigdata/zlin/Melanoma_meta/data/TNBC_Shiao/adata_cd45n.h5')
diopy.output.write_h5(adata_cd45p, file = '/bigdata/zlin/Melanoma_meta/data/TNBC_Shiao/adata_cd45p.h5')